In [18]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## CellOracle
import celloracle as co

## Load data

In [39]:
## DATA
adata = sc.read_h5ad("../data/data_diff_express_lncRNA.h5ad")
base_GRN = pd.read_parquet('../data/celloracle_data/base_GRN_edge_list.parquet')

## Modify data for CellOracle use:

NOTE: non-protein genes are not in adata since filtered. We see that apparently C13 is marked as protein coding!!

In [29]:
# Verify lncRNA presence in the dataset
print(f"'Rmst' in adata.var_names: {'Rmst' in adata.var_names}")
print(f"'Rmst' in adata.raw.var_names: {'Rmst' in adata.raw.var_names}")
print('    ')

print(f"'C13' in adata.var_names: {'C130026I21Rik' in adata.var_names}")
print(f"'C13' in adata.raw.var_names: {'C130026I21Rik' in adata.raw.var_names}")
print('    ')
print('    ')

print(f"Annotated gene biotype for Rmst: {adata.raw.var.loc['Rmst', 'gene_biotype']}") # we need to check in .raw
print('    ')

print(f"Annotated gene biotype for C13: {adata.var.loc['C130026I21Rik', 'gene_biotype']}")

'Rmst' in adata.var_names: False
'Rmst' in adata.raw.var_names: True
    
'C13' in adata.var_names: True
'C13' in adata.raw.var_names: True
    
    
Annotated gene biotype for Rmst: lincRNA
    
Annotated gene biotype for C13: protein_coding


In [ ]:

'''
# =============================================================
# PREPARE THE ANNDATA FOR CELLORACLE
# CellOracle needs raw counts or log-normalized in adata.X
# =============================================================

# We restore from adata.raw
adata_co = adata.copy()
adata_co.X = adata.raw[:, adata.var_names].X

# CellOracle requires the UMAP and cluster information:
# verify they exist
print("Embeddings available:", list(adata_co.obsm.keys()))
print('    ')
print("Clusters:\n",adata_co.obs['leiden'].value_counts())
print('    ')
print('    ')


# =============================================================
# CONVERT EDGE LIST TO CELLORACLE TFdict FORMAT
# TFdict structure: {target_gene: [TF1, TF2, ...]}
# =============================================================

# Filter base GRN to genes present in adata
genes_in_adata = set(adata_co.raw.var_names)

# We need to keep the lncRNAs (filtered in adata)
lncRNA_genes = {'Rmst', 'C130026I21Rik'}  

base_GRN_filtered = base_GRN[
    (base_GRN['target'].isin(genes_in_adata) | base_GRN['target'].isin(lncRNA_genes)) &
    (base_GRN['source'].isin(genes_in_adata) | base_GRN['source'].isin(lncRNA_genes))
]
'''

In [40]:
# =============================================================
# 1. DEFINE THE ACTIVE SUBSPACE (The Genes we need)
# =============================================================

# A. Extract all Unique Genes from the Network (Sources and Targets)
grn_genes = set(base_GRN['source'].unique()).union(set(base_GRN['target'].unique()))

# B. Extract the Topological Skeleton (HVGs from our previous UMAP)
hvg_genes = set(adata.var_names)

# C. The LncRNAs explicitly
lncrnas = {'Rmst', 'C130026I21Rik'}

# D. The Unified Universe
# We combine GRN genes, HVGs, and our lncRNAs.
target_universe = grn_genes.union(hvg_genes).union(lncrnas)

# E. Physical Intersection
# We must ensure all these genes ACTUALLY exist in the raw measurement matrix
final_genes_for_oracle = [g for g in target_universe if g in adata.raw.var_names]

print(f"Total theoretical genes requested: {len(target_universe)}")
print(f"Total genes physically present in .raw: {len(final_genes_for_oracle)}")

# =============================================================
# 2. CONSTRUCT THE CELLORACLE ANNDATA OBJECT
# =============================================================
print("\n--- 2. CONSTRUCTING adata_co ---")

# We extract the specific Log-Normalized data from .raw
# We create a BRAND NEW AnnData object to ensure clean memory and no Z-score contamination
adata_co = sc.AnnData(
    X = adata.raw[:, final_genes_for_oracle].X.copy(),
    obs = adata.obs.copy(),
    var = adata.raw[:, final_genes_for_oracle].var.copy()
)

# Crucial: CellOracle needs the spatial coordinates to project the simulation vectors
adata_co.obsm['X_pca'] = adata.obsm['X_pca'].copy()
adata_co.obsm['X_umap'] = adata.obsm['X_umap'].copy()

print(f"adata_co created successfully.")
print(f"Shape: {adata_co.shape} (Cells x Genes)")
print(f"Max Value (should be ~10, NOT Z-scored): {adata_co.X.max():.2f}")

# =============================================================
# 3. CONVERT EDGE LIST TO TFdict FORMAT
# =============================================================
print("\n--- 3. FILTERING GRN AND CREATING TFdict ---")

# Now we filter the GRN using our perfectly matched adata_co
genes_in_adata = set(adata_co.var_names)

base_GRN_filtered = base_GRN[
    (base_GRN['target'].isin(genes_in_adata)) &
    (base_GRN['source'].isin(genes_in_adata))
]

print(f"Edges before filtering to adata genes: {len(base_GRN)}")
print(f"Edges after filtering: {len(base_GRN_filtered)}")
print('    ')
print('    ')

# Convert to TFdict:
TFdict = (
    base_GRN_filtered
    .groupby('target')['source']
    .apply(list)
    .to_dict()
)

print(f"Target genes with at least one regulator: {len(TFdict)}")
print('    ')
print('    ')


## Sanity check: 
# Verify RMST1 edges are present
rmst1_as_regulator = {
    target: regs for target, regs in TFdict.items()
    if 'Rmst' in regs
}

# Verify C13 edges are present
c13_as_regulator = {
    target: regs for target, regs in TFdict.items()
    if 'C130026I21Rik' in regs
}
print(f"Genes regulated by 'Rmst' in TFdict: {len(rmst1_as_regulator)}")
print('    ')
print(f"Genes regulated by 'C13' in TFdict: {len(c13_as_regulator)}")

Total theoretical genes requested: 23712
Total genes physically present in .raw: 20122

--- 2. CONSTRUCTING adata_co ---
adata_co created successfully.
Shape: (1604, 20122) (Cells x Genes)
Max Value (should be ~10, NOT Z-scored): 6.55

--- 3. FILTERING GRN AND CREATING TFdict ---
Edges before filtering to adata genes: 6876574
Edges after filtering: 4849730
    
    
Target genes with at least one regulator: 17835
    
    
Genes regulated by 'Rmst' in TFdict: 0
    
Genes regulated by 'C13' in TFdict: 0


In [36]:
print(f"RMST1 edges: {sum(base_GRN_filtered['source']=='Rmst')}")

RMST1 edges: 608


## CellOracle inference

In [37]:
# =============================================================
# INITIALIZE ORACLE AND INJECT TFdict DIRECTLY
# =============================================================

oracle = co.Oracle()

oracle.import_anndata_as_raw_count(
    adata=adata_co,
    cluster_column_name='leiden',
    embedding_name='X_umap'
)

oracle.perform_PCA()
oracle.knn_imputation(n_pca_dims=19, k=20)

# Assign TFdict directly — bypasses import_TF_data entirely
oracle.TFdict = TFdict

print("TFdict assigned. Proceeding to fit GRN.")


# =============================================================
# STEP 5: FIT GRN — RIDGE REGRESSION PER CLUSTER
# This estimates the actual regulatory weights W_ij from
# expression co-variation, using oracle.TFdict as sparsity mask.
# alpha controls Ridge penalization (higher = sparser weights)
# =============================================================

oracle.fit_GRN_for_simulation(
    alpha=10,                          # Ridge regularization strength
    use_cluster_specific_TFdict=False  # use same prior for all clusters
)


# =============================================================
# STEP 6: SIMULATE RMST1 KNOCKOUT
# Sets Rmst expression to 0 in all cells and propagates
# the perturbation through the fitted GRN.
# =============================================================

oracle.simulate_shift(
    perturb_condition={"Rmst": 0.0},  # KO: set Rmst to zero
    n_propagation=3   # number of propagation steps through the network
)


# =============================================================
# STEP 7: COMPUTE TRANSITION PROBABILITIES AND EMBEDDING SHIFT
# Translates the gene expression shift into a probability of
# transitioning to neighboring cells in the UMAP embedding.
# =============================================================

oracle.estimate_transition_prob(
    n_neighbors=40,
    knn_random=True,
    sampled_fraction=0.5
)

oracle.calculate_embedding_shift(sigma_corr=0.05)


# =============================================================
# STEP 8: VISUALIZE VECTOR FIELD ON UMAP
# =============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Grid-based vector field (cleaner visualization)
oracle.plot_simulation_flow_on_grid(
    scale=0.4,
    ax=axes[0],
    color_by='leiden'
)
axes[0].set_title('RMST1 KO — vector field (grid)')

# Single-cell arrows
oracle.plot_simulation_flow_random_sampling(
    scale=0.4,
    ax=axes[1],
    color_by='leiden',
    n_each_cluster=30
)
axes[1].set_title('RMST1 KO — vector field (single cells)')

plt.tight_layout()
#plt.savefig('../figures/rmst1_ko_vector_field.pdf', dpi=150)
plt.show()


# =============================================================
# SAVE ORACLE OBJECT FOR FURTHER ANALYSIS
# =============================================================

#oracle.to_hdf5("../data/oracle_rmst1_ko.celloracle.hdf5")

20122 genes were found in the adata. Note that Celloracle is intended to use around 1000-3000 genes, so the behavior with this number of genes may differ from what is expected.
TFdict assigned. Proceeding to fit GRN.


  0%|          | 0/6 [00:00<?, ?it/s]

TypeError: Oracle_visualization.plot_simulation_flow_on_grid() got an unexpected keyword argument 'color_by'

Nanog example: we don't have Rmst included yet

In [4]:
# =============================================================
# STEP 6: SIMULATE RMST1 KNOCKOUT
# Sets Rmst expression to 0 in all cells and propagates
# the perturbation through the fitted GRN.
# =============================================================

oracle.simulate_shift(
    perturb_condition={"Nanog": 0.0},  # KO: set Rmst to zero
    n_propagation=3   # number of propagation steps through the network
)


# =============================================================
# STEP 7: COMPUTE TRANSITION PROBABILITIES AND EMBEDDING SHIFT
# Translates the gene expression shift into a probability of
# transitioning to neighboring cells in the UMAP embedding.
# =============================================================

oracle.estimate_transition_prob(
    n_neighbors=40,
    knn_random=True,
    sampled_fraction=0.5
)

oracle.calculate_embedding_shift(sigma_corr=0.05)

In [13]:
fig, ax = plt.subplots(1, 2,  figsize=[13, 6])

scale = 100
# Show quiver plot
oracle.plot_quiver(scale=scale, ax=ax[0])
ax[0].set_title(f"Simulated cell identity shift vector Nanog KO")

# Show quiver plot that was calculated with randomized graph.
oracle.plot_quiver_random(scale=scale, ax=ax[1])
ax[1].set_title(f"Randomized simulation vector")

plt.show()